Cell-level DEA (MAST or wilcoxon) per cell type. Here, one comparison is chosen from `comparisons`, cells are selected and subset to a brain region, and each cell type is tested separately. Output is one xlsx per contrast (one sheet per cell type, incl. group means, % expressing and cell/sample counts) and volcano plots. Paper settings and run commands: see README.

In [ ]:
import logging
import os
import re
import sys
from pathlib import Path

import anndata as ad
import xlsxwriter  # noqa: F401
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

repo = Path("/dss/dsshome1/0C/ra98gaq/git/cellseg-benchmark")
sys.path.insert(1, str(repo))
from cellseg_benchmark.dea_utils import add_ensembl_id, safe_sheet

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logger = logging.getLogger("dea")

In [ ]:
comp = "lowNeuN_ECKO_vs_Glut_WT"  # key in comparisons
method = "MAST-RE"         # MAST-RE (sample random effect), MAST-FE (no sample effect), wilcoxon_scanpy
low_pct = None             # lowNeuN WT percentile (5-25), None = 10%
dist = 18                  # EC-to-lowNeuN distance cutoff (um): 18, 31, 58, 103
region = "cortex"         # cortex, hippocampus, white_matter, None = all
max_cells = None           # max cells per condition (MAST only)
cdr = False                # cellular detection rate covariate (MAST only)
batch = None               # extra fixed covariate (MAST only), e.g. "slide"
pooled = True              # also test all cell types pooled (KO_vs_WT only)
subsets = None             # restrict to these cell types, e.g. ["Neurons_Glut"] (None = all)
joint = False              # ECKO vs WT comparisons: fit all genotypes (ECKO, WT, PCKO, GLKO), test ECKO only (needed with batch)
overwrite = True
seg = "Cellpose_1_nuclei_model"  # segmentation folder
f_neun = "20261015_temp-after-neuronal-subclustering-reannotated-fulldata.h5ad.gz"
f_ec = "20260302_temp-after_ec_proximal_to_lowNeuN.h5ad.gz"

genes = [
    "Acsl4", "Gclc", "Gclm", "Gpx4", "Slc7a11", "Slc11a2", "Tfrc", "Trf",
    "Foxf2", "Tek", "Vegfa", "Vegfb", "Cldn5", "Cdh5", "Tjp1", "Tjp2",
    "Nos3", "Sod1", "Cat", "Slc3a2", "Abcg2",
]

In [ ]:
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/analysis/foxf2") / seg
out = base / "dea"
prox = f"ec_proximal_to_lowNeuN_{dist}um"
regions = {"cortex": ["CTX"], "hippocampus": ["HIP", "DG-sg", "CAsp"], "white_matter": ["Fibre tract"]}

comparisons = {
    "KO_vs_WT": dict(
        file="adata_integrated.h5ad.gz", query="genotype in ['ECKO', 'WT']", cond="genotype", ref="WT", pooled=True,
        name="ECKO_vs_WT",
    ),
    "lowNeuN_vs_normalNeuN_ECKO": dict(
        file=f_neun, query="genotype == 'ECKO' and Rbfox3_low.notna()", cond="Rbfox3_low", ref="False",
        name="ECKO-lowNeuN_vs_ECKO-normalNeuN",
    ),
    "lowNeuN_vs_normalNeuN_WT": dict(
        file=f_neun, query="genotype == 'WT' and Rbfox3_low.notna()", cond="Rbfox3_low", ref="False",
        name="WT-lowNeuN_vs_WT-normalNeuN",
    ),
    "lowNeuN_ECKO_vs_Glut_WT": dict(
        file=f_neun, query="(genotype == 'ECKO' and Rbfox3_low == True) or (genotype == 'WT' and Rbfox3_low.notna())",
        cond="genotype", ref="WT",
        name="ECKO-lowNeuN_vs_WT-Glut",
    ),
    "ECs_proximal_vs_distal_ECKO": dict(
        file=f_ec, query="cell_type == 'ECs' and genotype == 'ECKO'", cond=prox, ref="False",
        name=f"ECKO-ECs-{dist}um-from-lowNeuN_vs_ECKO-other-ECs",
    ),
    "ECs_proximal_ECKO_vs_WT": dict(
        file=f_ec, query=f"cell_type == 'ECs' and ((genotype == 'ECKO' and {prox} == True) or genotype == 'WT')",
        cond="genotype", ref="WT",
        name=f"ECKO-ECs-{dist}um-from-lowNeuN_vs_WT-ECs",
    ),
}
if joint:
    comparisons["KO_vs_WT"]["query"] = "genotype in ['ECKO', 'WT', 'PCKO', 'GLKO']"
    comparisons["lowNeuN_ECKO_vs_Glut_WT"] |= dict(file="adata_integrated.h5ad.gz", query=(
        "(genotype == 'ECKO' and Rbfox3_low == True) or (genotype == 'WT' and Rbfox3_low.notna())"
        " or (genotype in ['PCKO', 'GLKO'] and cell_type == 'Neurons-Glut')"))
    comparisons["ECs_proximal_ECKO_vs_WT"] |= dict(file="adata_integrated.h5ad.gz", query=(
        f"cell_type == 'ECs' and ((genotype == 'ECKO' and {prox} == True) or genotype in ['WT', 'PCKO', 'GLKO'])"))
cfg = comparisons[comp]
cond, ref = cfg["cond"], cfg["ref"]
name = f"foxf2_{cfg['name']}_{region or 'all'}_{method}" + ("-CDR" if cdr and method.startswith("MAST") else "") + (f"-{batch}" if batch else "") + ("-joint" if joint and cond == "genotype" else "")
name += f"_{max_cells // 1000}k" if max_cells else ""
name += f"_lowNeuN-p{low_pct}" if low_pct not in (None, 10) else ""
run_dir = out / (method + ("-CDR" if cdr and method.startswith("MAST") else "") + (f"_{max_cells // 1000}k" if max_cells else ""))
run_dir.mkdir(exist_ok=True)
name

In [ ]:
adata = sc.read_h5ad(base / "adatas" / cfg["file"])
if cfg["file"] == "adata_integrated.h5ad.gz":
    sub = pd.read_parquet(base / "neuronal-subclustering" / "cell_type_sub_all_genotypes.parquet")["cell_type_sub"]
    adata.obs["cell_type"] = sub.reindex(adata.obs_names).fillna(adata.obs["cell_type_revised"].astype(str))  # neuronal labels from 02/03
if comp != "KO_vs_WT":
    v = pd.read_parquet(out / "sensitivity" / "lowneun_variants.parquet").filter(regex=rf"^p{low_pct or 10}\|")
    v.columns = v.columns.str.split("|").str[1]
    adata.obs[v.columns] = v.reindex(adata.obs_names)
for c in {"Rbfox3_low", prox} & set(adata.obs):
    adata.obs[c] = adata.obs[c].astype(str).map({"True": True, "False": False})
if cfg["query"]:
    adata = adata[adata.obs.query(cfg["query"], engine="python").index].copy()
if joint and comp == "lowNeuN_ECKO_vs_Glut_WT":
    adata.obs["cell_type"] = "Neurons-Glut"
if region:
    labels = pd.read_csv(base / "spatial_registration.csv", index_col=0)["label"]
    adata = adata[adata.obs_names.map(labels).isin(regions[region])].copy()
    print(f"{region}: {adata.n_obs} cells")

obs = adata.obs
obs["cell_type"] = (
    obs["cell_type"].astype(str)
    .replace({"Tanycytes": "Ependymal", "Astroependymal": "Astrocytes", "Neurons-Glyc-Gaba": "Neurons-Other"})
    .str.replace(r"[ /-]", "_", regex=True).str.replace(r"[+()]", "", regex=True)
)
obs[cond] = obs[cond].astype(str)
batch = batch if batch in obs else None
paired = (obs.groupby("sample", observed=True)[cond].nunique() > 1).any()
print(f"{cond}: {ref} (ref) vs {sorted(set(obs[cond]) - {ref})} | condition varies within samples: {paired} | batch: {batch}")
pd.crosstab(obs["cell_type"], [obs[cond], obs["sample"]])

In [ ]:
adata = ad.AnnData(
    X=sp.csr_matrix(adata.layers["volume_log1p_norm"], dtype=np.float32),
    obs=obs[["cell_type", cond, "sample"] + ([batch] if batch else [])].astype("category"),
    var=pd.DataFrame(index=adata.var_names),
)
groups = {ct: adata[adata.obs["cell_type"] == ct].copy() for ct in sorted(adata.obs["cell_type"].unique())}
if cfg.get("pooled") and pooled:
    groups["all"] = adata
if subsets:
    groups = {k: v for k, v in groups.items() if k in subsets}

In [ ]:
if method.startswith("MAST"):
    import anndata2ri
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter

    conv = ro.default_converter + pandas2ri.converter + anndata2ri.converter
    ro.r["source"](str(repo / "cellseg_benchmark" / "dea_utils.r"))
    ro.r(f"options(mc.cores = {os.environ.get('MC_CORES', max(1, len(os.sched_getaffinity(0)) - 1))})")
    mast_run_cached = ro.globalenv["mast_run_cached"]
    cache_dir = out / "mast_dea_cache" / name

In [ ]:
results = []
for ct, g in groups.items():
    n = g.obs[cond].value_counts()
    tests = [t for t in n.index if t != ref and n[t] >= 2 and t not in ("PCKO", "GLKO")]
    if n.get(ref, 0) < 2 or not tests:
        print(f"skip {ct}: {n.to_dict()}")
        continue
    print(f"--> {ct}: {n.to_dict()}")

    if method == "wilcoxon_scanpy":
        nz = (g.X > 0).astype(np.float32)
        det_any = np.max([np.asarray(nz[(g.obs[cond] == c).to_numpy()].mean(0)).ravel() for c in [ref, *tests]], axis=0)
        keep = (det_any >= 0.10) | (np.asarray(nz.mean(0)).ravel() >= 0.20)
        gg = g[:, keep].copy()
        gg.obs[cond] = gg.obs[cond].cat.remove_unused_categories()
        sc.tl.rank_genes_groups(gg, groupby=cond, groups=tests, reference=ref, method="wilcoxon", corr_method="benjamini-hochberg")
        res = pd.concat([sc.get.rank_genes_groups_df(gg, group=t).assign(test_group=t) for t in tests])
        res = res.rename(columns={"names": "gene", "logfoldchanges": "log2FC", "pvals": "PValue", "pvals_adj": "FDR"})
        res["FC"] = 2 ** res["log2FC"]
        res["design"] = f"~ {cond} (cell-level, no sample effect)"
    else:
        with localconverter(conv):
            r = mast_run_cached(
                adata=g, group_i=ct, test_groups=ro.StrVector(tests), ref_group=ref, condition_col=cond,
                sample_col="sample", batch_col=batch or ro.NULL, cache_dir=str(cache_dir), overwrite=overwrite,
                clear_cache=False, sample_random_effects=method == "MAST-RE",
                max_cells_per_condition=max_cells or ro.NULL, brain_region_subset=region or ro.NULL, cdr=cdr,
            )
            if r is ro.NULL:
                print(f"skip {ct}: MAST failed or too few cells (see R log in {cache_dir})")
                continue
            res = pd.DataFrame(ro.conversion.rpy2py(r)).rename(columns={"model": "design"})

    nz = (g.X > 0).astype(np.float32)
    m_ref = (g.obs[cond] == ref).to_numpy()
    stats = []
    for t in tests:
        m = (g.obs[cond] == t).to_numpy()
        stats.append(pd.DataFrame({
            "gene": g.var_names, "test_group": t,
            "test_group_n": g.obs.loc[m, "sample"].nunique(), "ref_n": g.obs.loc[m_ref, "sample"].nunique(),
            "n_cells_test": m.sum(), "n_cells_ref": m_ref.sum(),
            "mean_test": np.asarray(g.X[m].mean(0)).ravel(), "mean_ref": np.asarray(g.X[m_ref].mean(0)).ravel(),
            "pct_test": 100 * np.asarray(nz[m].mean(0)).ravel(), "pct_ref": 100 * np.asarray(nz[m_ref].mean(0)).ravel(),
        }))
    res["test_group"] = res["test_group"].astype(str)
    results.append(res.merge(pd.concat(stats), on=["gene", "test_group"], how="left").assign(subset=ct))

res = pd.concat(results, ignore_index=True)
res["method"], res["ref"], res["brain_region_subset"] = method, ref, region
res["test"] = res["test_group"] + "vs" + ref
order = [
    "subset", "brain_region_subset", "gene", "FC", "log2FC", "PValue", "FDR", "wald", "scores", "method", "design",
    "test_group", "ref", "test", "test_group_n", "ref_n", "n_cells_test", "n_cells_ref",
    "mean_test", "mean_ref", "pct_test", "pct_ref",
]
res = res[[c for c in order if c in res]].copy()
try:
    res = add_ensembl_id(res, logger=logger)
except Exception as e:  # no internet (mygene.info)
    print(f"no Ensembl IDs added: {e}")
res.sort_values("PValue").head(10)

In [ ]:
for test, d in res.groupby("test"):
    xlsx = run_dir / f"{name}.xlsx"
    if xlsx.exists() and not overwrite:
        print(f"exists, skip: {xlsx}")
        continue
    used = set()
    with pd.ExcelWriter(xlsx, engine="xlsxwriter") as w:
        for s, x in d.groupby("subset"):
            x.sort_values("PValue").to_excel(w, sheet_name=safe_sheet(s, used), index=False)
    print(f"wrote: {xlsx}")

In [ ]:
for test, d in res.groupby("test"):
    cts = sorted(d["subset"].unique())
    ncol = min(4, len(cts))
    nrow = -(-len(cts) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.5 * ncol, 4 * nrow), squeeze=False)
    for ax, s in zip(axes.flat, cts):
        x = d[d["subset"] == s]
        y = -np.log10(x["FDR"].clip(lower=1e-300))
        sig = x["FDR"] < 0.05
        ax.scatter(x.loc[~sig, "log2FC"], y[~sig], s=3, c="lightgrey")
        ax.scatter(x.loc[sig, "log2FC"], y[sig], s=3, c="tab:red")
        lab = x[x["gene"].isin(genes) | x.index.isin(x.nsmallest(10, "FDR").index)]
        for i, r in lab.iterrows():
            ax.text(r["log2FC"], y[i], r["gene"], fontsize=6, fontweight="bold" if r["gene"] in genes else "normal")
        ax.set_title(f"{s} (n sig = {sig.sum()})", fontsize=8)
        ax.set_xlabel("log2FC")
        ax.set_ylabel("-log10(FDR)")
    for ax in axes.flat[len(cts):]:
        ax.axis("off")
    fig.suptitle(name, fontsize=8)
    fig.tight_layout()
    fig.savefig(run_dir / f"{name}.png", bbox_inches="tight", dpi=200)
    plt.show()